## Load Libraries and Functions

In [2]:
import random
import pandas as pd
import numpy as np
import biovec
import json
import seaborn as sns
import matplotlib.pyplot as plt
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Seq import Seq
from itertools import product
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error as mae, r2_score as r2, root_mean_squared_error as rmse
from scipy.constants import micro as MICRO, nano as NANO, milli as MILLI, liter as LITER

/home/vtovmasian/miniconda3/envs/melt_temp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
random.seed(42)

results = pd.DataFrame(columns=["Nearest Neighbors", "Naive (XGB)", "Cross-strand (XGB)", "Duplex (XGB)", "Simple Encoding (NN)", "Naive (NN)", "Cross-strand (NN)", "Duplex (NN)", "Simple Encoding (NN)" ], index=["Valid Predictions", "Invalid Predictions", "MAE", "RMSE", "R2"])

In [4]:
def generate_random_dna_sequence(length: int) -> str:
    """Generate a random DNA sequence of given length."""
    return ''.join(np.random.choice(['A', 'T', 'C', 'G'], size=length))

def sample_dna_sequences(length: int, sample_size: int=4**8) -> list[str]:
    """Randomly sample DNA sequences of a given length with no duplicates."""
    max_seq = 4 ** length
    if sample_size > max_seq:
        raise ValueError(f"Cannot sample {sample_size} unique sequences, exceeds maximum possible sequences {max_seq}")
    
    # Generate unique sequences until sample size is reached.
    seqs = set()
    while len(seqs) < sample_size:
        seq = generate_random_dna_sequence(length)
        seqs.add(seq)
    return list(seqs)

def dna_complement(base: str) -> str:
    """Get the Watson-Crick complement of a DNA base."""
    complement = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C'
    }
    return complement.get(base, base)  # Return the base itself if not found

def random_mismatch_perturbation(sequence: str, num_mismatches: int=1) -> tuple[str, str]:
    """Introduce random mismatches in the sequence."""
    seq_list= list(sequence)
    comp_list = [dna_complement(base) for base in seq_list]
    length = len(seq_list)

    positions = np.random.choice(length, size=num_mismatches, replace=False) # Return numpy array of positions, 'replace' indicates no duplicates

    for i in positions:
        new_base = np.random.choice([b for b in 'ATCG' if b != dna_complement(seq_list[i])])
        comp_list[i] = new_base

    return ''.join(seq_list), ''.join(comp_list)

def generate_dna_sequences(length):
    """Generate a list of all DNA sequences of a given length."""
    return [''.join(seq) for seq in product('ATCG', repeat=length)]

## Data Loading / Preprocessing / Statistics

In [5]:
perfect_complement_data = pd.read_csv('dna_melt_temp.csv')

perfect_complement_data.head()

,Sequence,Oligo Concentration (M),Salt Concentration (M),pH,GC Content,Experimental Temperature
0,CCGG,0.0001,1.0,7,1.0000,16.6
1,CGCG,0.0001,1.0,7,1.0000,23.7
2,GCGC,0.0001,1.0,7,1.0000,27.5
3,CCGCGG,0.0001,1.0,7,1.0000,55.2
4,CGATCG,0.0001,1.0,7,0.6667,34.3


In [6]:
mismatch_data = pd.read_csv('olveira.csv')

mismatch_data.head()

,Unnamed: 0,i,Centre,Temp Exp,Temp Pred,Melt Index,Mismatches
0,0,1,GCG/CGC,69.3,71.47,3.8891,0
1,1,2,CGC/GCG,69.1,69.73,3.8559,0
2,2,3,GGC/CCG,68.9,69.73,3.8559,0
3,3,4,GCC/CGG,68.7,69.73,3.8559,0
4,4,5,CGG/GCC,68.2,70.50,3.8707,0


In [7]:
perfect_complement_data["Top Strand"] = perfect_complement_data["Sequence"]
perfect_complement_data["Bottom Strand"] = perfect_complement_data["Sequence"].apply(lambda x: str(Seq(x).complement()))
perfect_complement_data.drop(columns=['Sequence'], inplace=True)
perfect_complement_data["Mismatches"] = 0

perfect_complement_data.head()

,Oligo Concentration (M),Salt Concentration (M),pH,GC Content,Experimental Temperature,Top Strand,Bottom Strand,Mismatches
0,0.0001,1.0,7,1.0000,16.6,CCGG,GGCC,0
1,0.0001,1.0,7,1.0000,23.7,CGCG,GCGC,0
2,0.0001,1.0,7,1.0000,27.5,GCGC,CGCG,0
3,0.0001,1.0,7,1.0000,55.2,CCGCGG,GGCGCC,0
4,0.0001,1.0,7,0.6667,34.3,CGATCG,GCTAGC,0


In [8]:
# Clean Mismatch DataFrame by dropping unnecessary columns
mismatch_data.drop(columns=['Unnamed: 0', 'i', 'Melt Index', 'Temp Pred'], inplace=True, errors='ignore')

# Add Salt Concentration Column, pH column, and Oligo Concentration Column based on Olveira et al.
mismatch_data['Salt Concentration (M)'] = 0.06  # 50 mM
mismatch_data['pH'] = 7.4
mismatch_data['Oligo Concentration (M)'] = 0.000001  # 1.0 uM

# Add initial and terminal bases
initial_top = "CGACGTGC"
terminal_top = "ATGTGCTG"
initial_bot = str(Seq(initial_top).complement())
terminal_bot = str(Seq(terminal_top).complement())

mismatch_data["Top Strand"] = mismatch_data["Centre"].apply(lambda x: initial_top + str(x.split('/')[0]) + terminal_top)
mismatch_data["Bottom Strand"] = mismatch_data["Centre"].apply(lambda x: initial_bot + str(x.split('/')[1]) + terminal_bot)

# Drop Centre column
mismatch_data.drop(columns=["Centre"], inplace=True)

# Rename columns to match the perfect complement data
mismatch_data.rename(columns={'Temp Exp': 'Experimental Temperature'}, inplace=True)
mismatch_data["GC Content"] = (mismatch_data["Top Strand"].apply(lambda x: (x.count('G') + x.count('C'))) + mismatch_data["Bottom Strand"].apply(lambda x: (x.count('G') + x.count('C')))) / (len(mismatch_data["Top Strand"][0]) * 2)
mismatch_data.head()


,Experimental Temperature,Mismatches,Salt Concentration (M),pH,Oligo Concentration (M),Top Strand,Bottom Strand,GC Content
0,69.3,0,0.06,7.4,0.000001,CGACGTGCGCGATGTGCTG,GCTGCACGCGCTACACGAC,0.684211
1,69.1,0,0.06,7.4,0.000001,CGACGTGCCGCATGTGCTG,GCTGCACGGCGTACACGAC,0.684211
2,68.9,0,0.06,7.4,0.000001,CGACGTGCGGCATGTGCTG,GCTGCACGCCGTACACGAC,0.684211
3,68.7,0,0.06,7.4,0.000001,CGACGTGCGCCATGTGCTG,GCTGCACGCGGTACACGAC,0.684211
4,68.2,0,0.06,7.4,0.000001,CGACGTGCCGGATGTGCTG,GCTGCACGGCCTACACGAC,0.684211


In [9]:
# Combine datasets

data = pd.concat([perfect_complement_data, mismatch_data], ignore_index=True)


data.head()


,Oligo Concentration (M),Salt Concentration (M),pH,GC Content,Experimental Temperature,Top Strand,Bottom Strand,Mismatches
0,0.0001,1.0,7.0,1.0000,16.6,CCGG,GGCC,0
1,0.0001,1.0,7.0,1.0000,23.7,CGCG,GCGC,0
2,0.0001,1.0,7.0,1.0000,27.5,GCGC,CGCG,0
3,0.0001,1.0,7.0,1.0000,55.2,CCGCGG,GGCGCC,0
4,0.0001,1.0,7.0,0.6667,34.3,CGATCG,GCTAGC,0


In [10]:
data.tail()

,Oligo Concentration (M),Salt Concentration (M),pH,GC Content,Experimental Temperature,Top Strand,Bottom Strand,Mismatches
4731,0.000001,0.06,7.4,0.578947,42.2,CGACGTGCACCATGTGCTG,GCTGCACGATTTACACGAC,3
4732,0.000001,0.06,7.4,0.605263,42.1,CGACGTGCACCATGTGCTG,GCTGCACGCTTTACACGAC,3
4733,0.000001,0.06,7.4,0.631579,41.8,CGACGTGCACCATGTGCTG,GCTGCACGCTCTACACGAC,3
4734,0.000001,0.06,7.4,0.605263,41.6,CGACGTGCACTATGTGCTG,GCTGCACGCTCTACACGAC,3
4735,0.000001,0.06,7.4,0.631579,41.3,CGACGTGCACCATGTGCTG,GCTGCACGCCTTACACGAC,3


## Nearest Neighbors Analysis

In [11]:
def calculate_tm_safe(row):
    try:
        return mt.Tm_NN(row["Top Strand"], c_seq=row["Bottom Strand"], Na=row["Salt Concentration (M)"] / MILLI, saltcorr=7, dnac1=row["Oligo Concentration (M)"] / NANO, dnac2=row["Oligo Concentration (M)"])
    except:
        return np.nan

data["Nearest Neighbors"] = data.apply(calculate_tm_safe, axis=1)

Compute metrics and load into results

In [12]:

results.loc["Valid Predictions", "Nearest Neighbors"] = data["Nearest Neighbors"].notna().sum()
results.loc["Invalid Predictions", "Nearest Neighbors"] = data["Nearest Neighbors"].isna().sum()

nn = data.dropna(subset=["Nearest Neighbors", "Experimental Temperature"])

results.loc["MAE", "Nearest Neighbors"] = mae(nn["Experimental Temperature"], nn["Nearest Neighbors"])
results.loc["RMSE", "Nearest Neighbors"] = rmse(nn["Experimental Temperature"], nn["Nearest Neighbors"])
results.loc["R2", "Nearest Neighbors"] = r2(nn["Experimental Temperature"], nn["Nearest Neighbors"])

results.head()

,Nearest Neighbors,Naive (XGB),Cross-strand (XGB),Duplex (XGB),Simple Encoding (NN),Naive (NN),Cross-strand (NN),Duplex (NN),Simple Encoding (NN)
Valid Predictions,1896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Invalid Predictions,2840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MAE,2.133767,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RMSE,2.837837,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R2,0.921747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Protvec Corpi Generation

In [13]:
def generate_naive_corpus(max_length: int = 60, sample_size: int=4**8) -> tuple[list[str], list[str]]:
    """Generate a corpus of DNA sequences of varying lengths up to max_length."""
    corpus: list[str] = []
    corpus_mismatches: list[str] = []
    
    for length in tqdm(range(1, max_length + 1), desc="Generating DNA Corpus {length}", unit="length"):
        if length > 8:
            sequences = sample_dna_sequences(length, sample_size)
            mismatched_sequences = [random_mismatch_perturbation(seq, random.randint(0, 3))[1] for seq in sequences]
            corpus.extend(sequences)
            corpus_mismatches.extend(mismatched_sequences)
        else:
            sequences = generate_dna_sequences(length)
            mismatched_sequences = [random_mismatch_perturbation(seq, random.randint(0, 3) if length > 3 else 1)[1] for seq in sequences]
            corpus.extend(sequences)
            corpus_mismatches.extend(mismatched_sequences) 
    return corpus, corpus_mismatches

In [14]:
import os

# -----------------------------------------------------------------------------
# Configuration & Mismatch Mapping
# -----------------------------------------------------------------------------

# Mapping table for Approach 4 (Special Character Encoding)
# Watson-Crick pairs remain the top-strand letter; mismatches get explicit symbols.
MISMATCH_MAP = {
    ("G", "A"): "ɐ", ("A", "G"): "$",  # G/A tandem or single mismatch
    ("A", "A"): "Å",                   # A/A mismatch
    ("T", "T"): "⫧",                   # T/T mismatch
    ("C", "C"): "¢",                   # C/C mismatch
    ("G", "G"): "§",                   # G/G mismatch
    ("C", "A"): "@", ("A", "C"): "&",  # C/A mismatch
    ("T", "C"): "!", ("C", "T"): "¡",  # T/C mismatch
    ("T", "G"): "?", ("G", "T"): "¿",  # T/G mismatch
}

# -----------------------------------------------------------------------------
# Encoding Transformer Functions
# -----------------------------------------------------------------------------

def encode_approach_1(top: str, bottom: str, k: int = 3) -> str:
    """Approach 1: Duplex-Aware N-Grams (Concatenated k-mers across strands)"""
    tokens = []
    for i in range(len(top) - k + 1):
        top_k = top[i : i + k]
        bot_k = bottom[i : i + k]
        tokens.append(f"{top_k}-{bot_k}")
    return " ".join(tokens)

def encode_approach_2(top: str, bottom: str, k: int = 3) -> str:
    """Approach 2: Double Vector Representation (Separate FASTA entries for Top and Bottom)"""
    tokens = []
    for i in range(len(top) - k + 1):
        top_k = top[i : i + k]
        bot_k = bottom[i : i + k]
        tokens.append(f"{top_k}")
        tokens.append(f"{bot_k}")
    return " ".join(tokens)

def encode_approach_3(top: str, bottom: str) -> str:
    """Approach 3: Interleaved Strand Transformation (Top_i, Bot_i, Top_i+1...)"""
    interleaved = []
    for b_top, b_bot in zip(top, bottom):
        interleaved.append(b_top)
        interleaved.append(b_bot)
    return "".join(interleaved)


def encode_approach_4(top: str, bottom: str) -> str:
    """Approach 4: Mismatch Special Character Replacement"""
    encoded = []
    wc_pairs = {("A", "T"), ("T", "A"), ("C", "G"), ("G", "C")}
    
    for b_top, b_bot in zip(top, bottom):
        pair = (b_top.upper(), b_bot.upper())
        if pair in wc_pairs:
            encoded.append(b_top)  # Keep canonical top-strand base
        else:
            # Look up designated symbol or fallback to 'X'
            symbol = MISMATCH_MAP.get(pair, "X")
            encoded.append(symbol)
            
    return "".join(encoded)

# -----------------------------------------------------------------------------
# FASTA Corpus Generator
# -----------------------------------------------------------------------------

def build_fasta_corpora(duplex_dataset: list[tuple[str, str]], output_dir: str = "./fasta_corpora"):
    """
    Generates 4 FASTA files representing each duplex encoding strategy.
    
    Parameters:
        duplex_dataset: List of tuples containing (top_strand_5to3, bottom_strand_3to5).
                        Note: Strands MUST be aligned positionally 1-to-1.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    files = {
        1: open(os.path.join(output_dir, "approach_1_duplex_aware.fasta"), "w"),
        2: open(os.path.join(output_dir, "approach_2_double_vector.fasta"), "w"),
        3: open(os.path.join(output_dir, "approach_3_interleaved.fasta"), "w"),
        4: open(os.path.join(output_dir, "approach_4_special_char.fasta"), "w"),
    }
    
    for idx, (top, bottom) in enumerate(duplex_dataset):
        seq_id = f"duplex_{idx}"
        
        # Ensure aligned lengths match
        if len(top) != len(bottom):
            raise ValueError(f"Strand length mismatch at index {idx}: Top ({len(top)}) vs Bot ({len(bottom)})")

        # Approach 1: Duplex-Aware Token Stream (3-mers)
        app1_seq = encode_approach_1(top, bottom, k=3)
        files[1].write(f"{app1_seq}\n")
        
        # Approach 2: Double Vector (Separate FASTA entries for Top and Bottom)
        files[2].write(f"{top}\n")
        files[2].write(f"{bottom}\n")
        
        # Approach 3: Interleaved Strand
        app3_seq = encode_approach_3(top, bottom)
        files[3].write(f"{app3_seq}\n")
        
        # Approach 4: Special Character Mismatch
        app4_seq = encode_approach_4(top, bottom)
        files[4].write(f"{app4_seq}\n")

    # Close all open handles
    for f in files.values():
        f.close()
        
    print(f"Successfully generated 4 FASTA corpora in '{output_dir}/'")

In [15]:
top, bottom = generate_naive_corpus(60, sample_size=4**8)



Generating DNA Corpus {length}: 100%|██████████| 60/60 [04:14<00:00,  4.25s/length]


In [16]:
# Keep only strand pairs that are positionally length-matched
list_of_pairs = [(t, b) for t, b in zip(top, bottom) if len(t) == len(b)]

# Optional visibility into dropped misaligned pairs
dropped_pairs = min(len(top), len(bottom)) - len(list_of_pairs)
print(f"Using {len(list_of_pairs)} valid pairs; dropped {dropped_pairs} mismatched pairs.")

build_fasta_corpora(list_of_pairs, output_dir="./fasta_corpora")

Using 3495252 valid pairs; dropped 0 mismatched pairs.
Successfully generated 4 FASTA corpora in './fasta_corpora/'


In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from gensim import corpora

# -----------------------------
# Config
# -----------------------------
N_GRAM = 3
VEC_SIZE = 16
TEST_SIZE = 0.2
RANDOM_STATE = 42

fasta_dir = Path("./fasta_corpora")
model_dir = Path("./trained_models")
model_dir.mkdir(exist_ok=True)

fasta_files = {
    "approach_1_duplex_aware": fasta_dir / "approach_1_duplex_aware.fasta",
    "approach_2_double_vector": fasta_dir / "approach_2_double_vector.fasta",
    "approach_3_interleaved": fasta_dir / "approach_3_interleaved.fasta",
    "approach_4_special_char": fasta_dir / "approach_4_special_char.fasta",
}

# -----------------------------
# Train ProtVec models from existing FASTA corpora
# -----------------------------
protvec_models = {}
for name, fasta_path in fasta_files.items():
    if not fasta_path.exists():
        print(f"[skip] missing FASTA: {fasta_path}")
        continue

    model_path = model_dir / f"{name}.model"

    corpus = corpora.TextCorpus(str(fasta_path))  # Load the FASTA corpus for training

    pv = biovec.models.ProtVec(
        corpus=corpus,
        n=N_GRAM,
        size=VEC_SIZE,
        sg=1,
        window=7,
        min_count=1,
        workers=3
    )
    pv.save(str(model_path))
    protvec_models[name] = pv
    print(f"[ok] ProtVec trained: {name}")

# -----------------------------
# Feature builders
# -----------------------------
def vec_flat(pv, seq: str) -> np.ndarray:
    return pv.to_vecs(seq).reshape(-1)

def build_xy(df: pd.DataFrame, model_name: str, pv) -> tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    for row in df.itertuples(index=False):
        top = str(getattr(row, "Top Strand")).upper().strip()
        bottom = str(getattr(row, "Bottom Strand")).upper().strip()
        target = getattr(row, "Experimental Temperature")

        if len(top) != len(bottom) or len(top) < N_GRAM:
            continue
        if pd.isna(target):
            continue

        try:
            if model_name == "approach_1_duplex_aware":
                seq = encode_approach_1(top, bottom, k=N_GRAM)
                feat = vec_flat(pv, seq)
            elif model_name == "approach_2_double_vector":
                feat = np.concatenate([vec_flat(pv, top), vec_flat(pv, bottom)], axis=0)
            elif model_name == "approach_3_interleaved":
                seq = encode_approach_3(top, bottom)
                feat = vec_flat(pv, seq)
            elif model_name == "approach_4_special_char":
                seq = encode_approach_4(top, bottom)
                feat = vec_flat(pv, seq)
            else:
                continue

            X.append(feat)
            y.append(float(target))
        except Exception:
            # unseen n-grams or encoding/model mismatch
            continue

    if not X:
        return np.empty((0, 0)), np.empty((0,))
    return np.vstack(X), np.array(y)

# -----------------------------
# Train XGBoost per encoding
# -----------------------------
rows = []
for name, pv in protvec_models.items():
    X, y = build_xy(data, name, pv)
    valid = len(y)
    invalid = len(data) - valid

    if valid < 20:
        rows.append({
            "Encoding": name, "Valid Predictions": valid, "Invalid Predictions": invalid,
            "MAE": np.nan, "RMSE": np.nan, "R2": np.nan
        })
        print(f"[skip] {name}: not enough valid rows ({valid})")
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    reg = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=4
    )
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)

    rows.append({
        "Encoding": name,
        "Valid Predictions": valid,
        "Invalid Predictions": invalid,
        "MAE": mae(y_test, y_pred),
        "RMSE": rmse(y_test, y_pred),
        "R2": r2(y_test, y_pred),
    })

metrics_df = pd.DataFrame(rows).sort_values("RMSE")
display(metrics_df)

# Optional: write into your existing `results` table (matching current column names)
name_to_col = {
    "approach_2_double_vector": "Naive (XGB)",
    "approach_3_interleaved": "Cross-strand (XGB)",
    "approach_1_duplex_aware": "Duplex (XGB)",
}
for _, r in metrics_df.iterrows():
    col = name_to_col.get(r["Encoding"])
    if col in results.columns:
        results.loc["Valid Predictions", col] = r["Valid Predictions"]
        results.loc["Invalid Predictions", col] = r["Invalid Predictions"]
        results.loc["MAE", col] = r["MAE"]
        results.loc["RMSE", col] = r["RMSE"]
        results.loc["R2", col] = r["R2"]

display(results)

[ok] ProtVec trained: approach_1_duplex_aware
[ok] ProtVec trained: approach_2_double_vector
[ok] ProtVec trained: approach_3_interleaved
[ok] ProtVec trained: approach_4_special_char


AttributeError: 'Pandas' object has no attribute 'Top Strand'